# Train / Val Split — Great Barrier Reef Starfish Detection

Produces `data/splits.csv`, the canonical `image_id -> split` mapping used by
all downstream code (`dataset.py`, `model.py`).

**Design:** split is done per-video, at the sequence level (not a literal
whole-video holdout). With only 3 videos and very different empty-frame rates
(68% / 74.5% / 92%), holding out one whole video for val would be unfair —
val would be either near-empty or annotation-heavy relative to train.
Splitting each video's sequences independently (then combining) guarantees
every video is represented in both train and val, keeps each video's
empty/non-empty ratio consistent across the split, and still prevents
near-duplicate adjacent frames from leaking across train/val (a sequence is
always kept fully on one side).

**Known limitation:** video 2 has only 4 sequences, so its val split can't be
finely balanced by empty-rate — see the fairness report below.

**Usage in `model.py`:**
```python
train_csv = pd.read_csv('/content/data/train.csv')
train_csv['annotations'] = train_csv['annotations'].apply(ast.literal_eval)
splits = pd.read_csv('data/splits.csv')
full = train_csv.merge(splits[['image_id', 'split']], on='image_id')
train_data = full[full['split'] == 'train']
val_data = full[full['split'] == 'val']
```
Image path convention: `train_images/video_{video_id}/{video_frame}.jpg`,
where `video_id, video_frame = image_id.split('-')`.

In [1]:
from google.colab import userdata
import os

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

In [2]:
csv_path = '/content/data/train.csv'

if os.path.exists(csv_path):
    print(f'Found existing {csv_path}, skipping download')
else:
    os.makedirs('/content/data', exist_ok=True)
    os.system('kaggle competitions download -c tensorflow-great-barrier-reef -f train.csv -p /content/data')
    os.system('unzip -o /content/data/train.csv.zip -d /content/data')
    print(f'Downloaded and unzipped {csv_path}')

Downloaded and unzipped /content/data/train.csv


In [3]:
import ast
import pandas as pd

df = pd.read_csv(csv_path)
df['annotations'] = df['annotations'].apply(ast.literal_eval)
df['is_empty'] = df['annotations'].apply(len) == 0

# sequence IDs repeat across videos, so build a composite key
df['seq_group'] = df['video_id'].astype(str) + '_' + df['sequence'].astype(str)
print(f"Total sequences to split: {df['seq_group'].nunique()}")

Total sequences to split: 20


In [4]:
from sklearn.model_selection import GroupShuffleSplit

train_parts, val_parts = [], []

# split each video's sequences independently, then combine -- keeps every
# video represented on both sides and preserves each video's empty ratio
for vid in sorted(df['video_id'].unique()):
    video_df = df[df['video_id'] == vid]
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    tr_idx, va_idx = next(gss.split(video_df, groups=video_df['seq_group']))
    train_parts.append(video_df.iloc[tr_idx])
    val_parts.append(video_df.iloc[va_idx])

train_df = pd.concat(train_parts).reset_index(drop=True)
val_df = pd.concat(val_parts).reset_index(drop=True)

assert set(train_df['seq_group']).isdisjoint(val_df['seq_group']), \
    "Leak: a sequence appears in both splits"

In [5]:
print(f"Train: {len(train_df)} frames, {train_df['is_empty'].mean()*100:.1f}% empty")
print(f"Val:   {len(val_df)} frames, {val_df['is_empty'].mean()*100:.1f}% empty")

print("\nPer-video frame counts:")
print(pd.concat([
    train_df['video_id'].value_counts().rename('train'),
    val_df['video_id'].value_counts().rename('val'),
], axis=1))

print("\nPer-video empty %:")
print(pd.concat([
    (train_df.groupby('video_id')['is_empty'].mean() * 100).rename('train'),
    (val_df.groupby('video_id')['is_empty'].mean() * 100).rename('val'),
], axis=1).round(1))

Train: 19705 frames, 78.4% empty
Val:   3796 frames, 82.3% empty

Per-video frame counts:
          train   val
video_id             
2          7036  1525
1          6978  1254
0          5691  1017

Per-video empty %:
          train   val
video_id             
0          70.1  56.6
1          72.8  84.1
2          90.8  98.1


In [6]:
train_df['split'] = 'train'
val_df['split'] = 'val'
full_df = pd.concat([train_df, val_df]).reset_index(drop=True)
full_df = full_df[['image_id', 'video_id', 'sequence', 'split']]

os.makedirs('data', exist_ok=True)
full_df.to_csv('data/splits.csv', index=False)
print(full_df['split'].value_counts())

split
train    19705
val       3796
Name: count, dtype: int64
